
### Dataset Processing / Normalization

This Jupyter Notebook processes multiple datasets to prepare them for machine learning tasks. The idea is to prepare the data for the analysis on the thesis. We need to pre-process as the learning algorithms used requires numeric values. We also need to ensure the target and protected attributes are binary, as the majority of the metrics used require it. 

This code ensures that categorical variables are properly encoded for machine learning models while preserving the integrity of protected attributes. It also documents the transformations applied to the data for reproducibility.

In [6]:
import pandas as pd
import json
from termcolor import colored

In [7]:
dataset_names = ["Predict Students' Dropout and Academic Success",
                 "Employee dataset",
                 "COMPAS",
                 "Adult",
                 "Bank Marketing",
                 "Heart Dataset",
                 "Indian Liver Patient Dataset",
                 "AIDS Clinical Trials Group Study 175",
                 "Intersectional Bias Dataset",
                 "Diabetes",
                 "Glioma"]

dataset_files = [
    "datasets/dropout.csv",
    "datasets/Employee.csv",
    "datasets/compas-scores-raw.csv",
    "datasets/adult.csv",
    "datasets/bank-full.csv",
    "datasets/heart.csv",
    "datasets/indian.csv",
    "datasets/AIDS_ClinicalTrial_GroupStudy175.csv",
    "datasets/intersectional-bias.csv",
    "datasets/diabetes_binary_health_indicators_BRFSS2015.csv",
    "datasets/Glioma.csv",
]

dataset_targets = [
    "Target",
    "LeaveOrNot",
    "is_recid",
    "income",
    "y",
    "target",
    "Target",
    "label",
    "Diagnosis",
    "Diabetes_binary",
    "Grade"
]

datasets_favorable_outcomes = [
    'Graduate',
    0,
    0,
    '>50K',
    'yes',
    0,
    1,
    0,
    0,
    0.0,
    0
]


dataset_attr_mappings = [
    {"Gender": {"Female": 0, "Male": 1}},
    {"Gender": {"Female": 0, "Male": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"marital": {"not married": 0, "married": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"homo": {"Yes": 0, "No": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"Race": {"Non-White": 0, "White": 1}}
]

## Dataset information
Prints the metadata information for all the datasets. In this step we don't make any changes to the data, we simply display the raw information.

In [8]:
for name, file, protected_attr_mapping, target, favorable_outcome in zip(dataset_names, dataset_files, dataset_attr_mappings, dataset_targets, datasets_favorable_outcomes):
    df = pd.read_csv(file)
    protected_attr = list(protected_attr_mapping)[0]
    print(f"Dataset {colored(name, 'green')} has {colored(len(df), 'yellow')} instances, {colored(len(df.columns), 'cyan')} columns. Protected attr is {colored(protected_attr, 'red')}. Target is {colored(target, 'blue')} with values: {colored(list(df[target].value_counts().keys()), 'magenta')}")
    # plot protected attribute distribution

Dataset Predict Students' Dropout and Academic Success has 4424 instances, 37 columns. Protected attr is Gender. Target is Target with values: ['Graduate', 'Dropout', 'Enrolled']
Dataset Employee dataset has 4653 instances, 9 columns. Protected attr is Gender. Target is LeaveOrNot with values: [0, 1]
Dataset COMPAS has 11757 instances, 47 columns. Protected attr is sex. Target is is_recid with values: [0, 1, -1]
Dataset Adult has 32561 instances, 15 columns. Protected attr is sex. Target is income with values: ['<=50K', '>50K']
Dataset Bank Marketing has 45211 instances, 17 columns. Protected attr is marital. Target is y with values: ['no', 'yes']
Dataset Heart Dataset has 303 instances, 14 columns. Protected attr is sex. Target is target with values: [1, 0]
Dataset Indian Liver Patient Dataset has 583 instances, 11 columns. Protected attr is Sex. Target is Target with values: [1, 2]
Dataset AIDS Clinical Trials Group Study 175 has 2139 instances, 24 columns. Protected attr is homo. Ta

## Data Processing
In this step we prepare the data for the analysis.
We first drop all columns with Nan values, then we remove the duplicates.
We iterate through all the columns, when we hit a:
 1. Protected attribute: replace with the protected attributes mapping already defined;
 2. Target column: replace with the favorable outcome mapping to 1, else to 0 (for post training metrics definition); 
 3. Normal column with less than 10 unique values: one-hot-encode and add the new columns to the dataframe
 4. Normal column with more than 10 unique values: label-encode and add the column to the dataframe.

Then, the conversions are saved a json file named `{dataset_name}_conversions.json` and the data is written to `{dataset_name}_converted.csv`.
In this step, we both process and sanitize the data, as now we have everything in CSV, with the expected mappings and ready for training. 

In [9]:
converted_d_names = []
for dataset_name, dataset, protected_attr_mapping, target_attr, favorable_outcome in zip(dataset_names, dataset_files, dataset_attr_mappings, dataset_targets, datasets_favorable_outcomes):
    skipped_protected = False
    skipped_target = False
    protected_attr = list(protected_attr_mapping)[0]
    df = pd.read_csv(dataset)
    df.dropna(axis='columns', inplace=True)
    df.drop_duplicates(inplace=True)
    d_conversions = {}

    # if dataset is compas, drop columns name,first,last and handle timestamp columns
    if dataset_name == "COMPAS":
        df.drop(columns=['name', 'first', 'last'], inplace=True)
        date_columns = [
            "compas_screening_date",
            "dob",
            "v_screening_date",
            "screening_date",]
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].astype('int64') // 10**9

    for column in df.columns:
        if column == protected_attr:
            if dataset_name == "Glioma":
                # if map values > 0 to 1 and 0 to 0
                df[protected_attr] = df[protected_attr].map(lambda x: 1 if
                                                        x > 0 else 0)
            elif dataset_name == "Bank Marketing":
                # if map values 'married' to 1 and 'single','divorced','unknown' to 0
                df[protected_attr] = df[protected_attr].map(lambda x: 1 if
                                                        x == 'married' else 0)
            elif not pd.api.types.is_numeric_dtype(df[protected_attr]):
                df[protected_attr] = df[protected_attr].map(protected_attr_mapping[protected_attr])
            skipped_protected = True
            continue
        elif column == target_attr:
            df[column] = df[column].map(
                lambda x: 1 if x == favorable_outcome else 0)
            skipped_target = True
            continue
        elif pd.api.types.is_numeric_dtype(df[column]):
            continue

        dummies = pd.get_dummies(df[column], prefix=column, drop_first=False)
        # Drop original column and add dummy columns
        df = df.drop(column, axis=1)
        df = pd.concat([df, dummies], axis=1)
        # Save the mapping for reference
        d_conversions[column] = f"one_hot_encoded_to_{list(dummies.columns)}"
    if not skipped_protected:
        raise ValueError(
            f"Protected attribute '{protected_attr}' missing in dataset '{dataset} {df.columns} {protected_attr}'")
    if not skipped_target:
        raise ValueError(
            f"Target attribute '{target_attr}' missing in dataset '{dataset} {df.columns} {target_attr}'")

    df_name = dataset.split('/')[-1].split('.')[0]
    print(f"Dataset {colored(dataset_name, 'green').ljust(55)} has {colored(len(df), 'yellow').ljust(15)} instances, {colored(len(df.columns), 'cyan').ljust(12)} columns after encoding.")
    with open(f'datasets/{df_name}_conversions.json', 'w') as f:
        json.dump(d_conversions, f, indent=4)
    out_name = f'datasets/{df_name}_converted.csv'
    df.to_csv(out_name, index=False)
    converted_d_names.append(out_name)

Dataset Predict Students' Dropout and Academic Success has 4424   instances, 37  columns after encoding.
Dataset Employee dataset                               has 2764   instances, 14  columns after encoding.
Dataset COMPAS                                         has 11757  instances, 33  columns after encoding.
Dataset Adult                                          has 32537  instances, 108 columns after encoding.
Dataset Bank Marketing                                 has 45211  instances, 50  columns after encoding.
Dataset Heart Dataset                                  has 302    instances, 14  columns after encoding.
Dataset Indian Liver Patient Dataset                   has 570    instances, 11  columns after encoding.
Dataset AIDS Clinical Trials Group Study 175           has 2139   instances, 24  columns after encoding.
Dataset Intersectional Bias Dataset                    has 11000  instances, 24  columns after encoding.
Dataset Diabetes                                       

In [10]:
converted_d_names


['datasets/dropout_converted.csv',
 'datasets/Employee_converted.csv',
 'datasets/compas-scores-raw_converted.csv',
 'datasets/adult_converted.csv',
 'datasets/bank-full_converted.csv',
 'datasets/heart_converted.csv',
 'datasets/indian_converted.csv',
 'datasets/AIDS_ClinicalTrial_GroupStudy175_converted.csv',
 'datasets/intersectional-bias_converted.csv',
 'datasets/diabetes_binary_health_indicators_BRFSS2015_converted.csv',
 'datasets/Glioma_converted.csv']